# Datenaufbereitung (Data Preparation)
### Ziele der Datenvorbereitung
* **Datenanalyse:** Fokus auf **menschliche Interpretierbarkeit** (Saubere Beschriftungen, korrekte Daten, klare Diagramme).
* **Data Science:** Fokus auf **Maschinen-Bereitschaft** (Skalierung, Codierung, Feature Engineering).

In [ ]:
import pandas as pd

# Daten laden
df = pd.read_csv('data.csv')

# Erster Schritt: Das Chaos sichten
df.head(10)

### 🔍 Daten-Check: Erst schauen, dann coden!

Bevor wir zum nächsten Schritt übergehen, schau dir die Daten genau an. Du kannst die CSV-Datei direkt in **Excel** oder in **VS Code** (mit der *CSV Viewer* Extension) öffnen.

**Deine Aufgabe:**
1. Notiere dir, welche **Probleme** (Fehler, Lücken, Unstimmigkeiten) du in den Rohdaten beobachtest.
2. Überlege kurz, was wir tun könnten, um diese zu beheben (**Lösungsansätze**).

Komm anschließend hierher zurück. Wir werden die Probleme dann mit **Python-Funktionen** technisch verifizieren und die passenden Lösungen Schritt für Schritt umsetzen.

### 🔍 Initiales Daten-Audit
1. **Head:** `df.head()` für einen schnellen visuellen Scan.
2. **Datentypen:** `df.info()`, um falsche Formate zu finden.
3. **Ausreißer:** `df.describe()`, um unmögliche Min/Max-Werte zu entdecken.
4. **Lücken:** `df.isna().sum()`, um Datenlücken zu lokalisieren.
5. **Duplikate:** `df.duplicated().sum()`, um redundante Zeilen zu finden.
6. **Kategorien:** `.value_counts()`, um inkonsistente Labels (z. B. Groß-/Kleinschreibung) zu finden.

In [ ]:
# Die ersten zwei Zeilen anzeigen
df.head(2)

### Erste Eindrücke

- Insgesamt haben wir 6 Spalten (Timestamp, Symbol, Open, High, Low, Close, Volume).
- Ich muss beim Format des Zeitstempels (Timestamp) vorsichtig sein.
- Bezüglich Preis und Volumen könnten Ausreißer (Outlier) ein Problem darstellen.
- Es gibt Null-Werte (fehlende Daten).

Das Volumen zeigt mir den Handel pro Tag an. Ich weiß also nicht, wie viel gehandelt wurde, als der Preis hoch war. Ein kleiner Handel zu einem hohen Preis könnte meine Sicht auf den Preis signifikant verzerren, da er den Höchstpreis nach oben treibt.

In [ ]:
# Informationen zum Datensatz anzeigen
df.info()

- 'Close' hat 3 Null-Werte.
- 'Open' ist als 'Object' (Text) klassifiziert, sollte aber 'float' (Dezimalzahl) sein. Hier gibt es ein Problem – möglicherweise Text anstelle einer Zahl.

In [ ]:
# In numerische Werte umwandeln; ungültige Werte werden zu NaN (Not a Number)
num = pd.to_numeric(df['Open'], errors='coerce')

# Maske für die fehlerhaften Zeilen erstellen
mask = num.isna()

# Die problematischen Einträge anzeigen
df.loc[mask, 'Open']

In [ ]:
# Eintrag bei Index 9 anzeigen
df.loc[9]

In [ ]:
import numpy as np 
# Korrektur des 'Open'-Preises in Zeile 9
df.loc[9, 'Open'] = np.nan

# Oder mehrere Spalten in dieser Zeile gleichzeitig korrigieren
df.loc[9, ['Timestamp','Symbol','Open','High','Low','Close','Volume']] = [
    '2026-01-10', 'AMZN', np.nan, 165.9, 203.0, 146.7, 160.0
]

# Überprüfung
df.loc[9]

In [ ]:
import re
from datetime import datetime

def guess_type_and_format(s):
    s = str(s).strip()

    # --- DATUMSFORMATE ---
    date_formats = [
        ("%Y-%m-%d", r"^\d{4}-\d{2}-\d{2}$"),      # 1999-09-09
        ("%d/%m/%Y", r"^\d{2}/\d{2}/\d{4}$"),      # 09/12/2023
        ("%m/%d/%Y", r"^\d{2}/\d{2}/\d{4}$"),      # US-Stil
        ("%Y.%m.%d", r"^\d{4}\.\d{2}\.\d{2}$"),    # 2024.02.26
        ("%d.%m.%Y", r"^\d{2}\.\d{2}\.\d{4}$"),    # 22.02.2020
    ]

    for fmt, pattern in date_formats:
        if re.fullmatch(pattern, s):
            try:
                dt = datetime.strptime(s, fmt)
                return {"type": "Datum", "standard": dt.strftime("%Y-%m-%d")}
            except:
                continue

    # --- ZEITFORMATE ---
    time_formats = [
        ("%H:%M", r"^\d{2}:\d{2}$"),
        ("%H:%M:%S", r"^\d{2}:\d{2}:\d{2}$"),
    ]

    for fmt, pattern in time_formats:
        if re.fullmatch(pattern, s):
            try:
                dt = datetime.strptime(s, fmt)
                return {"type": "Zeit", "standard": dt.strftime("%H:%M:%S")}
            except:
                continue

    # --- NUMERISCH ---
    if s.isdigit():
        return {"type": "Integer", "standard": int(s)}

    try:
        return {"type": "Float", "standard": float(s)}
    except:
        pass

    return {"type": "String", "standard": s}

results = df["Timestamp"].astype(str).apply(guess_type_and_format)

df["type"] = results.apply(lambda x: x["type"])
df["Timestamp_standard"] = results.apply(lambda x: x["standard"])

In [ ]:
df.head(10)

In [ ]:
# Spalte 'Timestamp_standard' in echte Datetime-Objekte umwandeln
df['Timestamp_standard'] = pd.to_datetime(df['Timestamp_standard'], errors='coerce')

In [ ]:
df.head()

In [ ]:
df.info()

Es gibt **zwei Hauptwege**, um mit fehlenden Werten in einer Spalte umzugehen:
1. **Zeilen löschen** (Drop)
2. **Werte auffüllen** (Imputation)

In [ ]:
# Überprüfung der Indizes 8 bis 10; die Spalte 'Open' bei Index 9 ist NaN
df.loc[8:10]